In [ ]:
# Cell 1: Cài đặt thư viện cần thiết cho PIPNet, TorchLM, OpenCV & TensorBoard trên Google Colab / Kaggle
import os, sys, platform, subprocess

if platform.system() == "Linux":
    print("Đang cấu hình gói hệ thống trên Linux (Ubuntu/Colab)...")
    subprocess.run(
        "apt-get update -qq && apt-get install -y -qq libgl1 libglib2.0-0 >/dev/null 2>&1",
        shell=True
    )

print("Đang cài đặt các thư viện Python (torchlm, kagglehub, opencv, tensorboard, pyyaml, tqdm, scipy)...")
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "--no-warn-conflicts",
    "torchlm", "kagglehub", "opencv-python", "tensorboard", "tqdm", "pyyaml", "scipy"
])
print("Cài đặt hoàn tất! Môi trường đã sẵn sàng.")

In [2]:
# Cell 2: Tải bộ dữ liệu từ KaggleHub về thư mục cục bộ của Colab
import kagglehub
from pathlib import Path

# Tải phiên bản mới nhất của bộ dữ liệu video processed
path_origin = kagglehub.dataset_download(
    "nyvantran6634/data-processed-v3",
    output_dir="./kaggle/data-processed-v3"
)
print("Đường dẫn dữ liệu tải về:", path_origin)

# Tự động tìm thư mục chứa dataset_merged_split.csv
detected_dataset_dir = None
candidates = [
    Path(path_origin) / "data_processed",
    Path(path_origin),
    Path("./kaggle/data-processed-v3/data_processed"),
    Path("./kaggle/data-processed-v3")
]
for cand in candidates:
    if (cand / "dataset_merged_split.csv").is_file():
        detected_dataset_dir = cand
        break

if detected_dataset_dir is not None:
    print(f"Đã tìm thấy manifest dataset_merged_split.csv tại: {detected_dataset_dir}")
else:
    print(f"Đang quét tìm file manifest trong toàn bộ thư mục {path_origin}...")
    csv_files = list(Path(path_origin).rglob("dataset_merged_split.csv"))
    if csv_files:
        detected_dataset_dir = csv_files[0].parent
        print(f"Đã tìm thấy tại: {detected_dataset_dir}")
    else:
        detected_dataset_dir = Path(path_origin)
        print(f"Cảnh báo: Chưa tìm thấy dataset_merged_split.csv, mặc định dùng {detected_dataset_dir}")

Using Colab cache for faster access to the 'data-processed-v3' dataset.
Đường dẫn dữ liệu tải về: /kaggle/input/data-processed-v3
Đã tìm thấy manifest dataset_merged_split.csv tại: /kaggle/input/data-processed-v3/data_processed


In [3]:
# Cell 3: Import các thư viện cần thiết & Cấu hình môi trường
from __future__ import annotations
import csv
import dataclasses
from dataclasses import dataclass, field, asdict
import json
import logging
import math
import os
import random
import sys
import tempfile
import time
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple, Union

# Monkey patch cho scipy >= 1.14 để tương thích với torchlm metrics
import scipy.integrate
if not hasattr(scipy.integrate, "simps"):
    scipy.integrate.simps = scipy.integrate.simpson

# Đảm bảo mã hóa UTF-8 trên console
if hasattr(sys.stdout, "reconfigure"):
    try:
        sys.stdout.reconfigure(encoding="utf-8", errors="replace")
    except Exception:
        pass
if hasattr(sys.stderr, "reconfigure"):
    try:
        sys.stderr.reconfigure(encoding="utf-8", errors="replace")
    except Exception:
        pass

import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights
from torch.optim import Adam, AdamW
from torch.optim.lr_scheduler import LambdaLR
from torch.utils.data import DataLoader, Dataset
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm

# Cấu hình môi trường nạp DLL PyTorch CUDA (nếu chạy trên Windows/Linux)
torch_lib_path = os.path.join(os.path.dirname(sys.executable), "Library", "bin")
if os.path.exists(torch_lib_path):
    os.environ["PATH"] = torch_lib_path + os.pathsep + os.environ.get("PATH", "")
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

logger = logging.getLogger("DriverGuardian.LiteFat")
if not logger.handlers:
    handler = logging.StreamHandler(sys.stdout)
    handler.setFormatter(logging.Formatter("[%(levelname)s] %(name)s: %(message)s"))
    logger.addHandler(handler)
    logger.setLevel(logging.INFO)

In [4]:
# Cell 4: Hệ thống cấu hình trung tâm (TrainConfig)
# Không sử dụng cache, toàn bộ luồng huấn luyện chạy trực tiếp từ video gốc
@dataclass
class TrainConfig:
    """
    Cấu hình tổng thể cho pipeline huấn luyện mô hình LiteFat.
    Tất cả các siêu tham số được quản lý tập trung và phân nhóm rõ ràng.
    """
    # =====================================================================
    # 1. CẤU HÌNH KIẾN TRÚC MÔ HÌNH LITEFAT (Section III & IV)
    # =====================================================================
    num_nodes: int = 68
    feature_dim: int = 64
    mobilenet_proj_hidden: int = 192
    node_embed_dim: int = 16
    residual_channels: int = 64
    dilation_channels: int = 64
    skip_channels: int = 128
    end_channels: int = 256
    num_classes: int = 2
    num_blocks: int = 4
    layers_per_block: int = 2
    kernel_size: int = 2
    pretrained_backbone: bool = True
    dropout: float = 0.2
    freeze_backbone: bool = False
    freeze_backbone_epochs: int = 0  # Số epoch đầu đóng băng backbone trước khi unfreeze (0: tắt)

    # =====================================================================
    # 2. CẤU HÌNH TRÍCH XUẤT PIPNET & TIỀN XỬ LÝ (FAST PIPNET EXTRACTOR)
    # =====================================================================
    pipnet_backbone: str = "resnet18"
    letterbox_size: Tuple[int, int] = (640, 480)
    face_size: Tuple[int, int] = (224, 224)
    conf_threshold: float = 0.5
    nms_threshold: float = 0.3
    bbox_margin: float = 0.15
    box_smooth_alpha: float = 0.7

    # =====================================================================
    # 3. CẤU HÌNH BỘ DỮ LIỆU & LẤY MẪU THỜI GIAN (DATASET & TEMPORAL SAMPLING)
    # =====================================================================
    dataset_dir: str = "/kaggle/input/data-processed-v3/data_processed"
    manifest_filename: str = "dataset_merged_split.csv"
    time_interval: float = 0.1         # Lấy mẫu sau mỗi 0.1 giây (10 FPS)
    window_sec: Optional[float] = 10.0 # Cửa sổ trượt 10 giây (~100 khung hình)
    class_names: Dict[int, str] = field(default_factory=lambda: {
        0: "Alert",
        1: "Drowsy"
    })

    # =====================================================================
    # 4. CẤU HÌNH DATALOADER & HUẤN LUYỆN (TRAINING & OPTIMIZATION)
    # =====================================================================
    batch_size: int = 16
    num_workers: int = 0
    pin_memory: bool = True
    shuffle_train: bool = True
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    lr: float = 1e-4
    weight_decay: float = 1e-4
    epochs: int = 20
    warmup_epochs: int = 3
    min_lr_ratio: float = 0.05
    max_grad_norm: float = 1.0
    amp: bool = True
    label_smoothing: float = 0.0
    save_dir: str = "LiteFat/checkpoints"
    log_dir: Optional[str] = "LiteFat/logs"
    resume: Optional[str] = None
    seed: int = 42

    def __post_init__(self):
        """Đồng bộ và chuẩn hóa các giá trị cấu hình."""
        if not (0.0 <= float(self.label_smoothing) < 1.0):
            raise ValueError(f"label_smoothing phải nằm trong khoảng [0.0, 1.0), nhận được: {self.label_smoothing}")

        if isinstance(self.letterbox_size, int):
            self.letterbox_size = (self.letterbox_size, self.letterbox_size)
        elif isinstance(self.letterbox_size, (list, tuple)) and len(self.letterbox_size) == 2:
            self.letterbox_size = (int(self.letterbox_size[0]), int(self.letterbox_size[1]))

        if isinstance(self.face_size, int):
            self.face_size = (self.face_size, self.face_size)
        elif isinstance(self.face_size, (list, tuple)) and len(self.face_size) == 2:
            self.face_size = (int(self.face_size[0]), int(self.face_size[1]))

        if self.class_names is None or len(self.class_names) != self.num_classes:
            if self.num_classes == 2:
                self.class_names = {0: "Alert", 1: "Drowsy"}
            elif self.num_classes == 3:
                self.class_names = {0: "Alert", 1: "Fatigue", 2: "Drowsy"}
            else:
                self.class_names = {i: f"Class_{i}" for i in range(self.num_classes)}

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)

    def save(self, path: str | Path) -> None:
        p = Path(path)
        p.parent.mkdir(parents=True, exist_ok=True)
        data = self.to_dict()
        if p.suffix.lower() in (".yaml", ".yml"):
            try:
                import yaml
                with open(p, "w", encoding="utf-8") as f:
                    yaml.safe_dump(data, f, sort_keys=False, allow_unicode=True)
                return
            except ImportError:
                pass
        with open(p, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=4, ensure_ascii=False)

    @classmethod
    def load(cls, path: str | Path) -> "TrainConfig":
        p = Path(path)
        if not p.is_file():
            raise FileNotFoundError(f"Không tìm thấy file config tại: {p}")
        content = None
        if p.suffix.lower() in (".yaml", ".yml"):
            try:
                import yaml
                with open(p, "r", encoding="utf-8") as f:
                    content = yaml.safe_load(f)
            except ImportError:
                pass
        if content is None:
            with open(p, "r", encoding="utf-8") as f:
                content = json.load(f)
        if "class_names" in content and isinstance(content["class_names"], dict):
            content["class_names"] = {int(k): v for k, v in content["class_names"].items()}
        valid_fields = {f.name for f in dataclasses.fields(cls)}
        filtered_content = {k: v for k, v in content.items() if k in valid_fields}
        return cls(**filtered_content)

    def get_model_kwargs(self) -> Dict[str, Any]:
        return {
            "num_nodes": self.num_nodes,
            "feature_dim": self.feature_dim,
            "mobilenet_proj_hidden": self.mobilenet_proj_hidden,
            "node_embed_dim": self.node_embed_dim,
            "residual_channels": self.residual_channels,
            "dilation_channels": self.dilation_channels,
            "skip_channels": self.skip_channels,
            "end_channels": self.end_channels,
            "num_classes": self.num_classes,
            "num_blocks": self.num_blocks,
            "layers_per_block": self.layers_per_block,
            "kernel_size": self.kernel_size,
            "pretrained_backbone": self.pretrained_backbone,
            "dropout": self.dropout,
            "freeze_backbone": self.freeze_backbone,
        }

    def get_dataloader_kwargs(self) -> Dict[str, Any]:
        return {
            "dataset_dir": self.dataset_dir,
            "batch_size": self.batch_size,
            "num_workers": self.num_workers,
            "time_interval": self.time_interval,
            "window_sec": self.window_sec,
            "manifest_filename": self.manifest_filename,
            "letterbox_size": self.letterbox_size,
            "face_size": self.face_size,
            "conf_threshold": self.conf_threshold,
            "bbox_margin": self.bbox_margin,
            "pin_memory": self.pin_memory,
            "shuffle_train": self.shuffle_train,
            "device": self.device,
            "pipnet_backbone": self.pipnet_backbone,
            "nms_threshold": self.nms_threshold,
            "box_smooth_alpha": self.box_smooth_alpha,
        }

In [5]:
# Cell 5: Pipeline Tiền xử lý, Trích xuất PIPNet Trực tiếp & Nạp Dữ liệu (Không dùng Cache)
# =====================================================================
# 1. TIỆN ÍCH LETTERBOX & KHỞI TẠO PIPNET
# =====================================================================
def letterbox(
        image: np.ndarray,
        new_size: Optional[int | Tuple[int, int] | List[int]] = (640, 480),
        color: Tuple[int, int, int] = (114, 114, 114),
) -> np.ndarray:
    """Resize ảnh giữ nguyên tỷ lệ khung hình với đệm viền về kích thước đích (w, h)."""
    if isinstance(new_size, int):
        target_w, target_h = new_size, new_size
    else:
        target_w, target_h = new_size
    if target_w <= 0 or target_h <= 0:
        return image

    h, w = image.shape[:2]
    if w == target_w and h == target_h:
        return image

    scale = min(target_w / float(w), target_h / float(h))
    nw, nh = max(1, round(w * scale)), max(1, round(h * scale))
    interp = cv2.INTER_AREA if scale < 1.0 else cv2.INTER_LINEAR
    resized = cv2.resize(image, (nw, nh), interpolation=interp)

    if nw == target_w and nh == target_h:
        return resized

    canvas = np.full((target_h, target_w, 3), color, dtype=image.dtype)
    x, y = (target_w - nw) // 2, (target_h - nh) // 2
    canvas[y:y + nh, x:x + nw] = resized
    return canvas


def init_pipnet_68_gpu(backbone="resnet18"):
    """Khởi tạo FaceBoxesV2 + PIPNet (68 điểm 300W) trên GPU thông qua torchlm."""
    import torchlm
    from torchlm.tools import faceboxesv2
    from torchlm.models import pipnet

    device = "cuda" if torch.cuda.is_available() else "cpu"
    logger.info(f"Đang khởi tạo PIPNet ({backbone} - 68 điểm 300W) trên thiết bị: {device.upper()}")
    torchlm.runtime.bind(faceboxesv2(device=device))
    torchlm.runtime.bind(
        pipnet(
            backbone=backbone,
            pretrained=True,
            num_nb=10,
            num_lms=68,
            net_stride=32,
            input_size=256,
            meanface_type="300w",
            map_location=device
        )
    )
    logger.info("Đã nạp xong trọng số mô hình PIPNet 68!")


def compute_iou(box_a: np.ndarray, box_b: np.ndarray) -> float:
    """Tính Intersection over Union (IoU) giữa 2 hộp giới hạn [x1, y1, x2, y2]."""
    xa = max(box_a[0], box_b[0])
    ya = max(box_a[1], box_b[1])
    xb = min(box_a[2], box_b[2])
    yb = min(box_a[3], box_b[3])
    inter = max(0.0, xb - xa) * max(0.0, yb - ya)
    area_a = max(0.0, box_a[2] - box_a[0]) * max(0.0, box_a[3] - box_a[1])
    area_b = max(0.0, box_b[2] - box_b[0]) * max(0.0, box_b[3] - box_b[1])
    union = area_a + area_b - inter
    if union <= 0.0:
        return 0.0
    return float(inter / union)


# =====================================================================
# 2. BỘ TRÍCH XUẤT PIPNET TỐI ƯU HÓA GPU (FAST PIPNET EXTRACTOR)
# =====================================================================
class FastPIPNetExtractor:
    """
    Trích xuất 68 điểm mốc khuôn mặt và cắt khung hình cho LiteFat:
      - Batched FaceBoxesV2 trên GPU
      - Temporal Driver Tracking duy trì khuôn mặt tài xế
      - Batched PIPNet 300W giải mã NRM vector hóa
      - Landmark-Guided Box Refinement kết hợp EMA làm mượt
    """
    def __init__(
            self,
            backbone: str = "resnet18",
            device: Optional[str] = None,
            letterbox_size: Tuple[int, int] = (640, 480),
            face_size: Tuple[int, int] = (224, 224),
            conf_threshold: float = 0.5,
            nms_threshold: float = 0.3,
            bbox_margin: float = 0.15,
            box_smooth_alpha: float = 0.7,
    ):
        self.device = torch.device(
            device if device is not None else ("cuda" if torch.cuda.is_available() else "cpu")
        )
        self.letterbox_size = letterbox_size
        self.face_size = face_size
        self.conf_threshold = float(conf_threshold)
        self.nms_threshold = float(nms_threshold)
        self.bbox_margin = float(bbox_margin)
        self.box_smooth_alpha = float(box_smooth_alpha)

        from torchlm.runtime._wrappers import RuntimeWrapper
        if getattr(RuntimeWrapper, "face_base", None) is None or getattr(RuntimeWrapper, "landmarks_base", None) is None:
            init_pipnet_68_gpu(backbone=backbone)

        self.face_base = RuntimeWrapper.face_base
        self.face_net = self.face_base.net.to(self.device)
        self.face_net.eval()

        self.lmk_base = RuntimeWrapper.landmarks_base
        self.lmk_net = self.lmk_base.to(self.device)
        self.lmk_net.eval()

        target_w, target_h = self.letterbox_size
        from torchlm.tools._faceboxesv2 import PriorBox
        pbox = PriorBox(self.face_base.cfg, image_size=(target_h, target_w))
        self.cached_priors = pbox.forward().to(self.device)
        self.cached_variances = self.face_base.cfg["variance"]
        self.cached_scale = torch.tensor(
            [target_w, target_h, target_w, target_h], dtype=torch.float32, device=self.device
        )
        self.faceboxes_mean = torch.tensor(
            [104.0, 117.0, 123.0], dtype=torch.float32, device=self.device
        ).view(1, 3, 1, 1)

        self.pipnet_mean = torch.tensor(
            [0.485, 0.456, 0.406], dtype=torch.float32, device=self.device
        ).view(1, 3, 1, 1)
        self.pipnet_std = torch.tensor(
            [0.229, 0.224, 0.225], dtype=torch.float32, device=self.device
        ).view(1, 3, 1, 1)

        rev1 = self.lmk_base.reverse_index1
        rev2 = self.lmk_base.reverse_index2
        num_nb = self.lmk_base.num_nb
        self.rev_indices = torch.tensor(
            [r1 * num_nb + r2 for r1, r2 in zip(rev1, rev2)],
            dtype=torch.long,
            device=self.device,
        )

    @torch.no_grad()
    def _detect_faces_batched(self, frames: List[np.ndarray], max_chunk: int = 32) -> List[np.ndarray]:
        T = len(frames)
        if T == 0:
            return []
        all_dets: List[np.ndarray] = []
        priors_xy = self.cached_priors[None, :, :2]
        priors_wh = self.cached_priors[None, :, 2:]
        v0, v1 = self.cached_variances[0], self.cached_variances[1]

        for s in range(0, T, max_chunk):
            chunk = frames[s:s + max_chunk]
            cur_T = len(chunk)
            frames_np = np.stack(chunk, axis=0)
            tensor = torch.from_numpy(frames_np).permute(0, 3, 1, 2).to(self.device).float()
            tensor = tensor - self.faceboxes_mean

            loc, conf = self.face_net(tensor)
            scores = conf[:, :, 1]

            boxes_xy = priors_xy + loc[:, :, :2] * v0 * priors_wh
            boxes_wh = priors_wh * torch.exp(loc[:, :, 2:] * v1)
            b_x1y1 = boxes_xy - boxes_wh / 2.0
            b_x2y2 = b_x1y1 + boxes_wh
            decoded_boxes = torch.cat([b_x1y1, b_x2y2], dim=-1) * self.cached_scale

            for b in range(cur_T):
                sc = scores[b]
                mask = sc >= self.conf_threshold
                if not mask.any():
                    all_dets.append(np.empty((0, 5), dtype=np.float32))
                    continue
                bx = decoded_boxes[b, mask]
                sc_valid = sc[mask]
                keep = torchvision.ops.nms(bx, sc_valid, iou_threshold=self.nms_threshold)[:50]
                res = torch.cat([bx[keep], sc_valid[keep, None]], dim=-1).cpu().numpy()
                all_dets.append(res)
        return all_dets

    def _select_driver_faces(
            self,
            all_dets: List[np.ndarray],
            target_w: int,
            target_h: int
    ) -> Tuple[List[Optional[np.ndarray]], List[bool]]:
        T = len(all_dets)
        chosen_bboxes: List[Optional[np.ndarray]] = [None] * T
        found: List[bool] = [False] * T
        prev_box: Optional[np.ndarray] = None
        prev_box_area = 1.0

        for t in range(T):
            dets = all_dets[t]
            if len(dets) == 0:
                continue
            best_idx = 0
            if prev_box is None:
                best_score = -1.0
                for i, d in enumerate(dets):
                    area = max(0.0, d[2] - d[0]) * max(0.0, d[3] - d[1])
                    metric = d[4] * (area ** 0.5)
                    if metric > best_score:
                        best_score = metric
                        best_idx = i
            else:
                best_match = -1.0
                for i, d in enumerate(dets):
                    iou = compute_iou(d[:4], prev_box[:4])
                    area = max(0.0, d[2] - d[0]) * max(0.0, d[3] - d[1])
                    metric = (iou * 2.0) + (d[4] * 0.5) + (min(1.0, area / (prev_box_area + 1e-6)) * 0.3)
                    if metric > best_match:
                        best_match = metric
                        best_idx = i

            chosen = dets[best_idx].copy()
            chosen_bboxes[t] = chosen
            found[t] = True
            prev_box = chosen
            prev_box_area = max(1.0, (prev_box[2] - prev_box[0]) * (prev_box[3] - prev_box[1]))
        return chosen_bboxes, found

    @torch.no_grad()
    def _extract_landmarks_batched(
            self,
            frames: List[np.ndarray],
            chosen_bboxes: List[Optional[np.ndarray]],
            found: List[bool],
    ) -> Tuple[np.ndarray, np.ndarray]:
        T = len(frames)
        target_w, target_h = self.letterbox_size
        extend = 0.2
        valid_indices = [t for t in range(T) if found[t] and chosen_bboxes[t] is not None]

        landmarks_norm = np.ones((T, 68, 3), dtype=np.float32)
        landmarks_px = np.zeros((T, 68, 2), dtype=np.float32)

        if not valid_indices:
            return landmarks_norm, landmarks_px

        crop_coords, crops_resized = [], []
        for idx in valid_indices:
            bbox = chosen_bboxes[idx]
            bx1, by1, bx2, by2 = bbox[:4]
            bw = bx2 - bx1 + 1.0
            bh = by2 - by1 + 1.0
            cx1 = max(0, min(target_w - 2, int(bx1 - bw * extend / 2.0)))
            cy1 = max(0, min(target_h - 2, int(by1 + bh * extend / 2.0)))
            cx2 = max(cx1 + 1, min(target_w, int(bx2 + bw * extend / 2.0)))
            cy2 = max(cy1 + 1, min(target_h, int(by2 + bh * extend / 2.0)))

            crop_w = float(cx2 - cx1)
            crop_h = float(cy2 - cy1)
            crop_coords.append((cx1, cy1, crop_w, crop_h))

            face_crop = frames[idx][cy1:cy2, cx1:cx2, ::-1]
            face_resized = cv2.resize(face_crop, (256, 256), interpolation=cv2.INTER_LINEAR)
            crops_resized.append(face_resized)

        max_crop_chunk = 64
        batched_lms_list = []
        norm_factor = 1.0 * self.lmk_base.input_size / self.lmk_base.net_stride
        num_nb = self.lmk_base.num_nb
        max_len = self.lmk_base.max_len

        for s in range(0, len(crops_resized), max_crop_chunk):
            chunk_crops = crops_resized[s:s + max_crop_chunk]
            crops_np = np.stack(chunk_crops, axis=0)
            crops_tensor = torch.from_numpy(crops_np).permute(0, 3, 1, 2).to(self.device).float() / 255.0
            crops_tensor = (crops_tensor - self.pipnet_mean) / self.pipnet_std

            out_cls, out_x, out_y, out_nb_x, out_nb_y = self.lmk_net(crops_tensor)
            cur_B, num_lms, H_map, W_map = out_cls.shape
            out_cls = out_cls.view(cur_B, num_lms, -1)
            max_ids = torch.argmax(out_cls, dim=2)
            max_ids_col = max_ids.unsqueeze(-1)

            out_x_flat = out_x.view(cur_B, num_lms, -1)
            out_y_flat = out_y.view(cur_B, num_lms, -1)
            x_sel = torch.gather(out_x_flat, 2, max_ids_col).squeeze(2)
            y_sel = torch.gather(out_y_flat, 2, max_ids_col).squeeze(2)

            max_ids_nb = max_ids_col.repeat(1, 1, num_nb).view(cur_B, num_lms * num_nb, 1)
            out_nb_x_flat = out_nb_x.view(cur_B, num_lms * num_nb, -1)
            out_nb_y_flat = out_nb_y.view(cur_B, num_lms * num_nb, -1)
            nb_x_sel = torch.gather(out_nb_x_flat, 2, max_ids_nb).view(cur_B, num_lms, num_nb)
            nb_y_sel = torch.gather(out_nb_y_flat, 2, max_ids_nb).view(cur_B, num_lms, num_nb)

            pred_x = ((max_ids % W_map).float() + x_sel) / norm_factor
            pred_y = (torch.floor(max_ids.float() / W_map) + y_sel) / norm_factor
            pred_nb_x = ((max_ids % W_map).float().unsqueeze(-1) + nb_x_sel) / norm_factor
            pred_nb_y = (torch.floor(max_ids.float() / W_map).unsqueeze(-1) + nb_y_sel) / norm_factor

            pred_nb_x_flat = pred_nb_x.view(cur_B, -1)
            pred_nb_y_flat = pred_nb_y.view(cur_B, -1)
            nb_x_rev = pred_nb_x_flat[:, self.rev_indices].view(cur_B, num_lms, max_len)
            nb_y_rev = pred_nb_y_flat[:, self.rev_indices].view(cur_B, num_lms, max_len)

            final_x = torch.mean(torch.cat([pred_x.unsqueeze(-1), nb_x_rev], dim=-1), dim=-1)
            final_y = torch.mean(torch.cat([pred_y.unsqueeze(-1), nb_y_rev], dim=-1), dim=-1)
            chunk_lms = torch.stack([final_x, final_y], dim=-1).cpu().numpy()
            batched_lms_list.append(chunk_lms)

        batched_lms = np.concatenate(batched_lms_list, axis=0) if batched_lms_list else np.empty((0, 68, 2), dtype=np.float32)

        for i, idx in enumerate(valid_indices):
            cx1, cy1, cw, ch = crop_coords[i]
            score = float(chosen_bboxes[idx][4])
            lm_crop = batched_lms[i].copy()
            abs_x = lm_crop[:, 0] * cw + cx1
            abs_y = lm_crop[:, 1] * ch + cy1
            landmarks_px[idx, :, 0] = abs_x
            landmarks_px[idx, :, 1] = abs_y
            landmarks_norm[idx, :, 0] = abs_x / float(target_w)
            landmarks_norm[idx, :, 1] = abs_y / float(target_h)
            landmarks_norm[idx, :, 2] = score

        return landmarks_norm, landmarks_px

    def _crop_faces_for_litefat(
            self,
            frames: List[np.ndarray],
            chosen_bboxes: List[Optional[np.ndarray]],
            landmarks_px: np.ndarray,
            found: List[bool],
    ) -> torch.Tensor:
        T = len(frames)
        target_w, target_h = self.letterbox_size
        crop_w, crop_h = self.face_size
        raw_boxes = np.zeros((T, 4), dtype=np.float32)

        for t in range(T):
            if found[t] and chosen_bboxes[t] is not None:
                bx1, by1, bx2, by2 = chosen_bboxes[t][:4]
                lms = landmarks_px[t]
                lx1, ly1 = float(np.min(lms[:, 0])), float(np.min(lms[:, 1]))
                lx2, ly2 = float(np.max(lms[:, 0])), float(np.max(lms[:, 1]))
                fx1 = min(bx1, lx1)
                fy1 = min(by1, ly1)
                fx2 = max(bx2, lx2)
                fy2 = max(by2, ly2)
                fw = fx2 - fx1
                fh = fy2 - fy1
                cx1 = fx1 - fw * self.bbox_margin
                cy1 = fy1 - fh * self.bbox_margin
                cx2 = fx2 + fw * self.bbox_margin
                cy2 = fy2 + fh * self.bbox_margin
                center_x = (cx1 + cx2) / 2.0
                center_y = (cy1 + cy2) / 2.0
                side = max(cx2 - cx1, cy2 - cy1)
                raw_boxes[t] = [
                    center_x - side / 2.0, center_y - side / 2.0,
                    center_x + side / 2.0, center_y + side / 2.0
                ]

        valid_indices = [t for t in range(T) if found[t]]
        if not valid_indices:
            side = min(target_w, target_h) * 0.8
            cx, cy = target_w / 2.0, target_h / 2.0
            default_box = np.array([cx - side / 2.0, cy - side / 2.0, cx + side / 2.0, cy + side / 2.0], dtype=np.float32)
            for t in range(T):
                raw_boxes[t] = default_box
        else:
            first_idx = valid_indices[0]
            for t in range(first_idx):
                raw_boxes[t] = raw_boxes[first_idx]
            last_idx = valid_indices[-1]
            for t in range(last_idx + 1, T):
                raw_boxes[t] = raw_boxes[last_idx]
            for i in range(len(valid_indices) - 1):
                idx_a = valid_indices[i]
                idx_b = valid_indices[i + 1]
                if idx_b - idx_a > 1:
                    box_a = raw_boxes[idx_a]
                    box_b = raw_boxes[idx_b]
                    for t in range(idx_a + 1, idx_b):
                        alpha = float(t - idx_a) / float(idx_b - idx_a)
                        raw_boxes[t] = (1.0 - alpha) * box_a + alpha * box_b

        smoothed_boxes = np.zeros_like(raw_boxes)
        smoothed_boxes[0] = raw_boxes[0]
        alpha = self.box_smooth_alpha
        for t in range(1, T):
            smoothed_boxes[t] = alpha * raw_boxes[t] + (1.0 - alpha) * smoothed_boxes[t - 1]

        face_crops = np.zeros((T, crop_h, crop_w, 3), dtype=np.uint8)
        for t in range(T):
            bx1, by1, bx2, by2 = smoothed_boxes[t]
            x1 = max(0, min(target_w - 2, int(round(bx1))))
            y1 = max(0, min(target_h - 2, int(round(by1))))
            x2 = max(x1 + 1, min(target_w, int(round(bx2))))
            y2 = max(y1 + 1, min(target_h, int(round(by2))))
            crop = frames[t][y1:y2, x1:x2, :]
            if crop.size == 0 or crop.shape[0] < 2 or crop.shape[1] < 2:
                crop = frames[t]
            resized = cv2.resize(crop, (crop_w, crop_h), interpolation=cv2.INTER_LINEAR)
            face_crops[t] = resized[:, :, ::-1]

        face_tensor = torch.from_numpy(face_crops).permute(0, 3, 1, 2).float() / 255.0
        norm_mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
        norm_std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
        face_tensor = (face_tensor - norm_mean) / norm_std
        return face_tensor

    def extract_from_frames(
            self,
            frames: List[np.ndarray]
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        target_w, target_h = self.letterbox_size
        uniform_frames = []
        for f in frames:
            if f.shape[1] != target_w or f.shape[0] != target_h:
                uniform_frames.append(letterbox(f, new_size=(target_w, target_h)))
            else:
                uniform_frames.append(f)

        all_dets = self._detect_faces_batched(uniform_frames)
        chosen_bboxes, found = self._select_driver_faces(all_dets, target_w, target_h)
        landmarks_norm, landmarks_px = self._extract_landmarks_batched(uniform_frames, chosen_bboxes, found)
        face_frames_tensor = self._crop_faces_for_litefat(uniform_frames, chosen_bboxes, landmarks_px, found)
        landmarks_tensor = torch.from_numpy(landmarks_norm).float()
        return face_frames_tensor, landmarks_tensor


# =====================================================================
# 3. ĐỌC NHANH KHUNG HÌNH TỪ VIDEO
# =====================================================================
def fast_read_video_frames(
        video_path: Union[str, Path],
        target_indices: np.ndarray,
        letterbox_size: Tuple[int, int] = (640, 480),
) -> Tuple[List[np.ndarray], float, int]:
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise IOError(f"Không thể mở tệp video: {video_path}")

    fps = float(cap.get(cv2.CAP_PROP_FPS) or 30.0)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    target_w, target_h = letterbox_size

    frames: List[np.ndarray] = []
    curr_frame = int(target_indices[0])
    cap.set(cv2.CAP_PROP_POS_FRAMES, curr_frame)

    try:
        for target_idx in target_indices:
            target_idx = int(target_idx)
            if target_idx < curr_frame and frames:
                frames.append(frames[-1].copy())
                continue
            while curr_frame < target_idx:
                if not cap.grab():
                    break
                curr_frame += 1
            ret, frame = cap.read()
            if not ret or frame is None:
                break
            lb_frame = letterbox(frame, new_size=(target_w, target_h))
            frames.append(lb_frame)
            curr_frame += 1
    finally:
        cap.release()

    if not frames:
        raise IOError(f"Không đọc được khung hình nào từ: {video_path}")
    while len(frames) < len(target_indices):
        frames.append(frames[-1].copy())
    return frames, fps, total_frames


# =====================================================================
# 4. LỚP LITEFAT DATASET (TRÍCH XUẤT TRỰC TIẾP ON-THE-FLY)
# =====================================================================
class LiteFatDataset(Dataset):
    """
    PyTorch Dataset nạp dữ liệu trực tiếp cho mô hình LiteFat (không sử dụng cache):
      - Đọc video theo time_interval và window_sec.
      - Trích xuất 68 điểm mốc bằng FastPIPNetExtractor (Batched GPU).
      - Cắt khung hình khuôn mặt 224x224 chuẩn hóa ImageNet cho MobileNetV3-Small.
    """
    def __init__(
            self,
            dataset_dir: Union[str, Path] = "./kaggle/data-processed-v3/data_processed",
            split: str = "train",
            time_interval: float = 0.1,
            window_sec: Optional[float] = 10.0,
            manifest_filename: str = "dataset_merged_split.csv",
            letterbox_size: Tuple[int, int] = (640, 480),
            face_size: Tuple[int, int] = (224, 224),
            conf_threshold: float = 0.5,
            nms_threshold: float = 0.3,
            bbox_margin: float = 0.15,
            box_smooth_alpha: float = 0.7,
            pipnet_backbone: str = "resnet18",
            extractor: Optional[FastPIPNetExtractor] = None,
            device: Optional[str] = None,
    ):
        split = split.lower().strip()
        if split not in ("train", "val", "test"):
            raise ValueError(f"Split không hợp lệ: '{split}'. Chỉ chấp nhận 'train', 'val', hoặc 'test'.")

        self.root = Path(dataset_dir)
        self.split = split
        self.time_interval = float(time_interval) if time_interval is not None else 0.1
        self.window_sec = float(window_sec) if window_sec is not None else None
        self.manifest_filename = manifest_filename
        self.letterbox_size = letterbox_size
        self.face_size = face_size
        self.conf_threshold = conf_threshold
        self.bbox_margin = bbox_margin

        self.samples = self._load_samples()
        if len(self.samples) == 0:
            logger.warning(f"Không tìm thấy mẫu video nào trong {self.root} cho split='{self.split}'")

        if extractor is not None:
            self.extractor = extractor
        else:
            self.extractor = FastPIPNetExtractor(
                backbone=pipnet_backbone,
                device=device,
                letterbox_size=self.letterbox_size,
                face_size=self.face_size,
                conf_threshold=self.conf_threshold,
                nms_threshold=nms_threshold,
                bbox_margin=self.bbox_margin,
                box_smooth_alpha=box_smooth_alpha,
            )

    def _load_samples(self) -> List[Dict[str, Any]]:
        manifest_path = self.root / self.manifest_filename
        samples = []
        if manifest_path.is_file():
            with open(manifest_path, mode="r", encoding="utf-8-sig") as f:
                reader = csv.DictReader(f)
                for row in reader:
                    if row.get("split", "").strip().lower() == self.split:
                        rel_path = row.get("rel_path") or row.get("processed_path")
                        if not rel_path:
                            continue
                        norm_rel = Path(rel_path.replace("\\", "/")).as_posix()
                        full_path = self.root / norm_rel
                        samples.append({
                            "key": norm_rel,
                            "path": full_path,
                            "label": int(row["label"]),
                            "video_id": row.get("video_id") or Path(rel_path).stem,
                            "fps": float(row.get("fps", 30.0)),
                            "frame_count": int(float(row.get("frame_count", 0))),
                            "duration_sec": float(row.get("duration_sec", 0.0)),
                        })
        else:
            for label, folder in ((0, "0_alert"), (1, "1_drowsy")):
                split_folder = self.root / self.split / folder
                if split_folder.is_dir():
                    for p in sorted(split_folder.glob("*.mp4")):
                        samples.append({
                            "key": p.relative_to(self.root).as_posix(),
                            "path": p,
                            "label": label,
                            "video_id": p.stem,
                            "fps": 30.0,
                            "frame_count": 0,
                            "duration_sec": 0.0,
                        })
        return samples

    def _get_frame_indices(self, total_frames: int, fps: float) -> np.ndarray:
        if total_frames <= 0:
            return np.array([0], dtype=int)
        fps = fps if fps > 0 else 30.0
        duration_sec = total_frames / fps

        if self.time_interval == 0:
            if self.window_sec is None:
                return np.arange(total_frames, dtype=int)
            else:
                win_sec = min(self.window_sec, duration_sec)
                num_consec = int(round(win_sec * fps))
                num_consec = max(1, min(num_consec, total_frames))
                max_start = max(0, total_frames - num_consec)
                start_idx = random.randint(0, max_start) if (self.split == "train" and max_start > 0) else max_start // 2
                return np.arange(start_idx, start_idx + num_consec, dtype=int)

        if self.window_sec is None:
            timestamps = np.arange(0, duration_sec, self.time_interval)
            if len(timestamps) == 0:
                timestamps = np.array([0.0])
            indices = np.rint(timestamps * fps).astype(int)
            return np.clip(indices, 0, total_frames - 1)
        else:
            win = min(self.window_sec, duration_sec)
            max_start = max(0.0, duration_sec - win)
            start_sec = random.uniform(0.0, max_start) if (self.split == "train" and max_start > 0) else max_start / 2.0
            timestamps = start_sec + np.arange(0, win, self.time_interval)
            if len(timestamps) == 0:
                timestamps = np.array([start_sec])
            indices = np.rint(timestamps * fps).astype(int)
            return np.clip(indices, 0, total_frames - 1)

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, index: int) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        sample = self.samples[index]
        fps = sample["fps"]
        total_frames = sample["frame_count"]

        if fps <= 0 or total_frames <= 0:
            cap = cv2.VideoCapture(str(sample["path"]))
            fps = float(cap.get(cv2.CAP_PROP_FPS) or 30.0)
            total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 300)
            cap.release()
            sample["fps"] = fps
            sample["frame_count"] = total_frames

        target_indices = self._get_frame_indices(total_frames, fps)
        raw_frames, actual_fps, _ = fast_read_video_frames(
            sample["path"], target_indices, letterbox_size=self.letterbox_size
        )
        face_frames, landmarks = self.extractor.extract_from_frames(raw_frames)
        label = torch.tensor(sample["label"], dtype=torch.long)
        return face_frames, landmarks, label


# =====================================================================
# 5. COLLATE VÀ XÂY DỰNG DATALOADERS
# =====================================================================
def collate_litefat(batch: List[Tuple[torch.Tensor, torch.Tensor, torch.Tensor]]):
    face_frames_list = [item[0] for item in batch]
    landmarks_list = [item[1] for item in batch]
    labels_list = [item[2] for item in batch]

    lengths = [f.shape[0] for f in face_frames_list]
    max_len = max(lengths)

    padded_frames, padded_landmarks = [], []
    for f, lm, cur_len in zip(face_frames_list, landmarks_list, lengths):
        if cur_len < max_len:
            pad_t = max_len - cur_len
            last_frame = f[-1:].repeat(pad_t, 1, 1, 1)
            padded_f = torch.cat([f, last_frame], dim=0)
            pad_lm = torch.ones((pad_t, lm.shape[1], lm.shape[2]), dtype=lm.dtype)
            padded_l = torch.cat([lm, pad_lm], dim=0)
            padded_frames.append(padded_f)
            padded_landmarks.append(padded_l)
        else:
            padded_frames.append(f)
            padded_landmarks.append(lm)

    batch_frames = torch.stack(padded_frames, dim=0)       # [B, T, 3, 224, 224]
    batch_landmarks = torch.stack(padded_landmarks, dim=0) # [B, T, 68, 3]
    batch_labels = torch.stack(labels_list, dim=0)         # [B]
    return batch_frames, batch_landmarks, batch_labels


def build_litefat_dataloaders(
        dataset_dir: Union[str, Path] = "./kaggle/data-processed-v3/data_processed",
        batch_size: int = 8,
        num_workers: int = 0,
        time_interval: float = 0.1,
        window_sec: Optional[float] = 10.0,
        letterbox_size: Tuple[int, int] = (640, 480),
        face_size: Tuple[int, int] = (224, 224),
        manifest_filename: str = "dataset_merged_split.csv",
        pin_memory: bool = True,
        shuffle_train: bool = True,
        device: Optional[str] = None,
        pipnet_backbone: str = "resnet18",
        conf_threshold: float = 0.5,
        nms_threshold: float = 0.3,
        bbox_margin: float = 0.15,
        box_smooth_alpha: float = 0.7,
) -> Tuple[DataLoader, DataLoader, DataLoader]:
    shared_extractor = FastPIPNetExtractor(
        backbone=pipnet_backbone,
        device=device,
        letterbox_size=letterbox_size,
        face_size=face_size,
        conf_threshold=conf_threshold,
        nms_threshold=nms_threshold,
        bbox_margin=bbox_margin,
        box_smooth_alpha=box_smooth_alpha,
    )

    common_cfg = dict(
        dataset_dir=dataset_dir,
        time_interval=time_interval,
        window_sec=window_sec,
        manifest_filename=manifest_filename,
        letterbox_size=letterbox_size,
        face_size=face_size,
        conf_threshold=conf_threshold,
        nms_threshold=nms_threshold,
        bbox_margin=bbox_margin,
        box_smooth_alpha=box_smooth_alpha,
        extractor=shared_extractor,
        device=device,
    )

    train_ds = LiteFatDataset(split="train", **common_cfg)
    val_ds = LiteFatDataset(split="val", **common_cfg)
    test_ds = LiteFatDataset(split="test", **common_cfg)

    train_loader = DataLoader(
        train_ds, batch_size=batch_size, shuffle=shuffle_train,
        num_workers=num_workers, pin_memory=pin_memory, collate_fn=collate_litefat
    )
    val_loader = DataLoader(
        val_ds, batch_size=batch_size, shuffle=False,
        num_workers=num_workers, pin_memory=pin_memory, collate_fn=collate_litefat
    )
    test_loader = DataLoader(
        test_ds, batch_size=batch_size, shuffle=False,
        num_workers=num_workers, pin_memory=pin_memory, collate_fn=collate_litefat
    )
    return train_loader, val_loader, test_loader

In [6]:
# Cell 6: Kiến trúc Mô hình LiteFat (IEEE/RSJ IROS 2025) & Hàm Mục Tiêu LiteFatLoss
# =====================================================================
# 1. NHÁNH NGỮ CẢNH: MOBILENETV3-SMALL (Section III-C1)
# =====================================================================
class MobileNetV3ContextEncoder(nn.Module):
    """Trích xuất vector ngữ cảnh d in R^{D} cho mỗi khung hình khuôn mặt."""
    def __init__(
            self,
            out_dim: int = 64,
            proj_hidden_dim: int = 192,
            pretrained: bool = True,
            dropout: float = 0.2,
            freeze_backbone: bool = False
    ):
        super().__init__()
        weights = MobileNet_V3_Small_Weights.DEFAULT if pretrained else None
        base_model = mobilenet_v3_small(weights=weights)

        self.features = base_model.features
        self.avgpool = base_model.avgpool
        self.projector = nn.Sequential(
            nn.Linear(576, proj_hidden_dim),
            nn.Hardswish(inplace=True),
            nn.Dropout(p=dropout),
            nn.Linear(proj_hidden_dim, out_dim)
        )
        if freeze_backbone:
            self.freeze_backbone(freeze=True)

    def freeze_backbone(self, freeze: bool = True, freeze_projector: bool = False) -> None:
        for param in self.features.parameters():
            param.requires_grad = not freeze
        if freeze_projector:
            for param in self.projector.parameters():
                param.requires_grad = not freeze
        elif not freeze:
            for param in self.projector.parameters():
                param.requires_grad = True

    def unfreeze_backbone(self) -> None:
        self.freeze_backbone(freeze=False, freeze_projector=False)

    @property
    def is_backbone_frozen(self) -> bool:
        return not any(p.requires_grad for p in self.features.parameters())

    def train(self, mode: bool = True):
        super().train(mode)
        if mode and self.is_backbone_frozen:
            for m in self.features.modules():
                if isinstance(m, (nn.BatchNorm2d, nn.SyncBatchNorm)):
                    m.eval()
        return self

    def forward(self, frame_tensor: torch.Tensor) -> torch.Tensor:
        feat = self.features(frame_tensor)
        feat = self.avgpool(feat)
        feat = torch.flatten(feat, 1)
        return self.projector(feat)


# =====================================================================
# 2. MÔ-ĐUN DUNG HỢP ĐA PHƯƠNG THỨC (Section III-C2, Equation 1)
# =====================================================================
class MultimodalFusionModule(nn.Module):
    """Thực hiện phép dung hợp đa phương thức: X = (C * w) * d^T"""
    def __init__(self):
        super().__init__()
        self.w = nn.Parameter(torch.ones(3, 1) / 3.0)

    @staticmethod
    def handle_missing_landmarks(coords: torch.Tensor, conf_threshold: float = 1e-4) -> torch.Tensor:
        conf_sum = coords[..., 2].abs().sum(dim=-1, keepdim=True).unsqueeze(-1)
        invalid_mask = conf_sum < conf_threshold
        return torch.where(invalid_mask, torch.ones_like(coords), coords)

    def forward(self, coords: torch.Tensor, d: torch.Tensor) -> torch.Tensor:
        coords = self.handle_missing_landmarks(coords)
        node_scalar = torch.matmul(coords, self.w)
        d_row = d.unsqueeze(2)
        X = torch.matmul(node_scalar, d_row)
        return X.permute(0, 3, 2, 1).contiguous()


# =====================================================================
# 3. MA TRẬN KỀ TỰ THÍCH NGHI & TÍCH CHẬP ĐỒ THỊ (Section III-D1, Eq. 2 & 3)
# =====================================================================
class SelfAdaptiveAdjacency(nn.Module):
    def __init__(self, num_nodes: int = 68, embed_dim: int = 16):
        super().__init__()
        self.E1 = nn.Parameter(torch.randn(num_nodes, embed_dim) * 0.1)
        self.E2 = nn.Parameter(torch.randn(num_nodes, embed_dim) * 0.1)

    def forward(self) -> torch.Tensor:
        spatial_weights = F.relu(torch.matmul(self.E1, self.E2.transpose(0, 1)))
        return F.softmax(spatial_weights, dim=-1)


class SpatialGCNLayer(nn.Module):
    def __init__(self, in_dim: int = 64, hidden_dim: int = 64, out_dim: int = 64, dropout: float = 0.2):
        super().__init__()
        self.W0 = nn.Linear(in_dim, hidden_dim, bias=False)
        self.W1 = nn.Linear(hidden_dim, out_dim, bias=False)
        self.dropout = nn.Dropout(p=dropout)

    def forward(self, x: torch.Tensor, A_adp: torch.Tensor) -> torch.Tensor:
        x_perm = x.permute(0, 3, 2, 1).contiguous()
        h1 = self.W0(x_perm)
        h1 = torch.einsum("ij,btnf->btif", A_adp, h1)
        h1 = F.relu(h1)
        h1 = self.dropout(h1)

        h2 = self.W1(h1)
        h2 = torch.einsum("ij,btnf->btif", A_adp, h2)
        z = F.softmax(h2, dim=-1)
        return z.permute(0, 3, 2, 1).contiguous()


# =====================================================================
# 4. TÍCH CHẬP THỜI GIAN NHÂN QUẢ CÓ CỔNG (Section III-D2, Eq. 4 & 5)
# =====================================================================
class GatedDilatedCausalTCN(nn.Module):
    def __init__(self, in_channels: int = 64, dilation_channels: int = 64, kernel_size: int = 2, dilation: int = 1):
        super().__init__()
        self.causal_padding = (kernel_size - 1) * dilation
        self.theta_1 = nn.Conv2d(in_channels, dilation_channels, kernel_size=(1, kernel_size), dilation=(1, dilation), bias=True)
        self.theta_2 = nn.Conv2d(in_channels, dilation_channels, kernel_size=(1, kernel_size), dilation=(1, dilation), bias=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x_pad = F.pad(x, (self.causal_padding, 0, 0, 0))
        tcn_a = torch.tanh(self.theta_1(x_pad))
        tcn_b = torch.sigmoid(self.theta_2(x_pad))
        return tcn_a * tcn_b


# =====================================================================
# 5. KHỐI KHÔNG GIAN - THỜI GIAN (STGL BLOCK)
# =====================================================================
class STGLBlock(nn.Module):
    def __init__(self, residual_channels: int = 64, dilation_channels: int = 64, skip_channels: int = 128, kernel_size: int = 2, dilation: int = 1, dropout: float = 0.2):
        super().__init__()
        self.gated_tcn = GatedDilatedCausalTCN(residual_channels, dilation_channels, kernel_size=kernel_size, dilation=dilation)
        self.spatial_gcn = SpatialGCNLayer(dilation_channels, residual_channels, residual_channels, dropout=dropout)
        self.skip_conv = nn.Conv2d(dilation_channels, skip_channels, kernel_size=(1, 1))
        self.bn = nn.BatchNorm2d(residual_channels)

    def forward(self, x: torch.Tensor, A_adp: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        residual = x
        h_tcn = self.gated_tcn(x)
        skip = self.skip_conv(h_tcn)
        h_gcn = self.spatial_gcn(h_tcn, A_adp)
        x_res = self.bn(h_gcn + residual)
        return x_res, skip


# =====================================================================
# 6. KIẾN TRÚC TỔNG THỂ LITEFAT & LOSS
# =====================================================================
class LiteFat(nn.Module):
    def __init__(
            self,
            num_nodes: int = 68,
            feature_dim: int = 64,
            mobilenet_proj_hidden: int = 192,
            node_embed_dim: int = 16,
            residual_channels: int = 64,
            dilation_channels: int = 64,
            skip_channels: int = 128,
            end_channels: int = 256,
            num_classes: int = 2,
            num_blocks: int = 4,
            layers_per_block: int = 2,
            kernel_size: int = 2,
            pretrained_backbone: bool = True,
            dropout: float = 0.2,
            freeze_backbone: bool = False
    ):
        super().__init__()
        self.num_nodes = num_nodes
        self.feature_dim = feature_dim
        self.num_classes = num_classes

        self.context_encoder = MobileNetV3ContextEncoder(
            out_dim=feature_dim,
            proj_hidden_dim=mobilenet_proj_hidden,
            pretrained=pretrained_backbone,
            dropout=dropout,
            freeze_backbone=freeze_backbone
        )
        self.fusion = MultimodalFusionModule()
        self.adaptive_adj = SelfAdaptiveAdjacency(num_nodes=num_nodes, embed_dim=node_embed_dim)
        self.start_conv = nn.Conv2d(feature_dim, residual_channels, kernel_size=(1, 1))

        self.stgl_blocks = nn.ModuleList()
        for _ in range(num_blocks):
            dilation = 1
            for _ in range(layers_per_block):
                self.stgl_blocks.append(
                    STGLBlock(
                        residual_channels=residual_channels,
                        dilation_channels=dilation_channels,
                        skip_channels=skip_channels,
                        kernel_size=kernel_size,
                        dilation=dilation,
                        dropout=dropout
                    )
                )
                dilation *= 2

        self.end_conv_1 = nn.Conv2d(skip_channels, end_channels, kernel_size=(1, 1))
        self.end_conv_2 = nn.Conv2d(end_channels, num_classes, kernel_size=(1, 1))

    def forward_stgl_from_fused(self, X: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        A_adp = self.adaptive_adj()
        h = self.start_conv(X)
        skip_sum = None
        for block in self.stgl_blocks:
            h, skip = block(h, A_adp)
            skip_sum = skip if skip_sum is None else (skip_sum + skip)

        out = F.relu(skip_sum)
        out = F.relu(self.end_conv_1(out))
        out = self.end_conv_2(out)  # [B, num_classes, N, T]

        frame_logits = out.mean(dim=2).permute(0, 2, 1).contiguous()  # [B, T, num_classes]
        clip_logits = frame_logits.mean(dim=1)                        # [B, num_classes]
        return frame_logits, clip_logits

    def forward(
            self,
            video_frames: torch.Tensor,
            landmarks: torch.Tensor
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        B, T, C_img, H, W = video_frames.shape
        frames_flat = video_frames.view(B * T, C_img, H, W)
        d_flat = self.context_encoder(frames_flat)
        d = d_flat.view(B, T, self.feature_dim)
        X = self.fusion(landmarks, d)
        return self.forward_stgl_from_fused(X)

    def freeze_backbone(self, freeze: bool = True, freeze_projector: bool = False) -> None:
        self.context_encoder.freeze_backbone(freeze=freeze, freeze_projector=freeze_projector)

    def unfreeze_backbone(self) -> None:
        self.context_encoder.unfreeze_backbone()

    @property
    def is_backbone_frozen(self) -> bool:
        return self.context_encoder.is_backbone_frozen


class LiteFatLoss(nn.Module):
    """Hàm mục tiêu huấn luyện theo Phương trình (6) bài báo LiteFat."""
    def __init__(self, label_smoothing: float = 0.0):
        super().__init__()
        self.ce = nn.CrossEntropyLoss(reduction="mean", label_smoothing=label_smoothing)

    def forward(self, frame_logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        B, T, C = frame_logits.shape
        if targets.dim() == 1:
            targets = targets.unsqueeze(1).expand(B, T)
        return self.ce(frame_logits.reshape(B * T, C), targets.reshape(B * T))

In [7]:
# Cell 7: Lớp Quản Lý Huấn Luyện LiteFatTrainer (OOP)
# Theo dõi Loss, Accuracy, Recall, F1, F2 và lưu toàn bộ checkpoints (F1 quyết định best)
# =====================================================================
# 1. TIỆN ÍCH HỆ THỐNG & HÀM ĐỘC LẬP
# =====================================================================
def seed_everything(seed: int = 42) -> None:
    """Cố định seed để đảm bảo khả năng tái lập kết quả thực nghiệm."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def configure_logging(log_file: Path | str) -> None:
    """Cấu hình logging đồng thời ra Console và file train.log."""
    if hasattr(sys.stdout, "reconfigure"):
        try:
            sys.stdout.reconfigure(encoding="utf-8", errors="replace")
        except Exception:
            pass

    log_file = Path(log_file)
    log_file.parent.mkdir(parents=True, exist_ok=True)

    logger.setLevel(logging.INFO)
    logger.propagate = False
    for handler in logger.handlers[:]:
        handler.close()
        logger.removeHandler(handler)

    formatter = logging.Formatter("[%(asctime)s][%(levelname)s] %(message)s", datefmt="%Y-%m-%d %H:%M:%S")

    file_handler = logging.FileHandler(log_file, encoding="utf-8")
    file_handler.setFormatter(formatter)
    logger.addHandler(file_handler)

    stream_handler = logging.StreamHandler(sys.stdout)
    stream_handler.setFormatter(formatter)
    logger.addHandler(stream_handler)


def atomic_save(state: Any, path: str | Path) -> None:
    """Lưu checkpoint nguyên tử (Atomic Save) qua file tạm để tránh corrupt."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, tmp_file = tempfile.mkstemp(dir=path.parent, prefix=f".{path.name}.", suffix=".tmp")
    try:
        with os.fdopen(fd, "wb") as f:
            torch.save(state, f)
            f.flush()
            os.fsync(f.fileno())
        os.replace(tmp_file, path)
    finally:
        Path(tmp_file).unlink(missing_ok=True)


def get_cosine_warmup_scheduler(
        optimizer: torch.optim.Optimizer,
        total_epochs: int,
        warmup_epochs: int = 3,
        min_lr_ratio: float = 0.05,
) -> LambdaLR:
    """Tạo bộ điều tốc Cosine Annealing kèm Linear Warmup."""
    warmup_epochs = max(0, min(warmup_epochs, total_epochs - 1))

    def lr_lambda(epoch: int) -> float:
        if epoch < warmup_epochs:
            return float(epoch + 1) / float(max(1, warmup_epochs))
        progress = float(epoch - warmup_epochs) / float(max(1, total_epochs - warmup_epochs - 1))
        progress = min(1.0, max(0.0, progress))
        return min_lr_ratio + (1.0 - min_lr_ratio) * 0.5 * (1.0 + math.cos(math.pi * progress))

    return LambdaLR(optimizer, lr_lambda=lr_lambda)


def compute_metrics(
        y_true: np.ndarray,
        y_pred: np.ndarray,
        avg_loss: float,
        num_classes: int = 2,
) -> Dict[str, float]:
    """
    Tính toán chi tiết các chỉ số phân loại: Loss, Accuracy, Recall, F1, F2, Precision.
    - Bài toán 2 lớp: tính theo lớp dương (1: Drowsy).
    - Bài toán đa lớp: tính Macro-average.
    """
    total = len(y_true)
    if total == 0:
        return {
            "loss": float(avg_loss),
            "accuracy": 0.0,
            "precision": 0.0,
            "recall": 0.0,
            "f1": 0.0,
            "f2": 0.0,
            "n_samples": 0,
        }

    accuracy = float((y_true == y_pred).sum() / total)

    if num_classes == 2:
        tp = int(((y_pred == 1) & (y_true == 1)).sum())
        fp = int(((y_pred == 1) & (y_true == 0)).sum())
        fn = int(((y_pred == 0) & (y_true == 1)).sum())

        precision = float(tp / max(1, tp + fp))
        recall = float(tp / max(1, tp + fn))
        f1 = float((2 * tp) / max(1, 2 * tp + fp + fn))
        f2 = float((5 * tp) / max(1, 5 * tp + 4 * fn + fp))
    else:
        precisions, recalls, f1s, f2s = [], [], [], []
        for c in range(num_classes):
            tp_c = int(((y_pred == c) & (y_true == c)).sum())
            fp_c = int(((y_pred == c) & (y_true != c)).sum())
            fn_c = int(((y_pred != c) & (y_true == c)).sum())

            p_c = tp_c / max(1, tp_c + fp_c)
            r_c = tp_c / max(1, tp_c + fn_c)
            f1_c = (2 * p_c * r_c) / max(1e-12, p_c + r_c)
            f2_c = (5 * p_c * r_c) / max(1e-12, 4 * p_c + r_c)

            precisions.append(p_c)
            recalls.append(r_c)
            f1s.append(f1_c)
            f2s.append(f2_c)

        precision = float(np.mean(precisions))
        recall = float(np.mean(recalls))
        f1 = float(np.mean(f1s))
        f2 = float(np.mean(f2s))

    return {
        "loss": float(avg_loss),
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "f2": f2,
        "n_samples": total,
    }


def run_epoch(
        model: nn.Module,
        loader: DataLoader,
        criterion: nn.Module,
        device: torch.device,
        optimizer: Optional[torch.optim.Optimizer] = None,
        scaler: Optional[torch.amp.GradScaler] = None,
        amp: bool = True,
        max_grad_norm: float = 1.0,
        desc: str = "Train",
        num_classes: int = 2,
) -> Dict[str, float]:
    """Chạy 1 epoch qua DataLoader và trả về bộ chỉ số tổng hợp."""
    training = optimizer is not None
    model.train(training)

    total_loss = 0.0
    total_samples = 0
    all_targets = []
    all_preds = []

    progress = tqdm(loader, desc=desc, unit="batch", dynamic_ncols=True, leave=False)

    for batch in progress:
        video_frames, landmarks, labels = batch[0], batch[1], batch[2]
        video_frames = video_frames.to(device, non_blocking=True)
        landmarks = landmarks.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        count = labels.size(0)

        with torch.set_grad_enabled(training):
            with torch.autocast(device_type=device.type, enabled=amp and device.type == "cuda"):
                frame_logits, clip_logits = model(video_frames, landmarks)
                loss = criterion(frame_logits, labels)

            if not torch.isfinite(loss):
                raise FloatingPointError(f"Gặp giá trị Loss không hợp lệ (NaN/Inf) tại {desc}")

            if training:
                optimizer.zero_grad(set_to_none=True)
                if scaler is not None and scaler.is_enabled():
                    scaler.scale(loss).backward()
                    scaler.unscale_(optimizer)
                    if max_grad_norm > 0:
                        nn.utils.clip_grad_norm_(model.parameters(), max_norm=max_grad_norm)
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    loss.backward()
                    if max_grad_norm > 0:
                        nn.utils.clip_grad_norm_(model.parameters(), max_norm=max_grad_norm)
                    optimizer.step()

        preds = clip_logits.detach().argmax(dim=-1)
        total_loss += loss.detach().item() * count
        total_samples += count

        all_targets.append(labels.detach().cpu().numpy())
        all_preds.append(preds.cpu().numpy())

        current_loss = total_loss / max(1, total_samples)
        progress.set_postfix(loss=f"{current_loss:.4f}")

    if total_samples == 0:
        return {
            "loss": 0.0, "accuracy": 0.0, "precision": 0.0,
            "recall": 0.0, "f1": 0.0, "f2": 0.0, "n_samples": 0
        }

    y_true = np.concatenate(all_targets, axis=0)
    y_pred = np.concatenate(all_preds, axis=0)
    avg_loss = total_loss / total_samples
    return compute_metrics(y_true, y_pred, avg_loss, num_classes=num_classes)


def log_epoch_to_tensorboard(
        writer: SummaryWriter,
        train_metrics: Dict[str, float],
        val_metrics: Dict[str, float],
        current_lr: float,
        epoch: int,
) -> None:
    """Ghi nhận chi tiết các scalar Loss, Accuracy, Recall, F1, F2 cho cả Train & Val."""
    for metric_name in ("loss", "accuracy", "recall", "f1", "f2", "precision"):
        if metric_name in train_metrics:
            writer.add_scalar(f"train/{metric_name}", train_metrics[metric_name], epoch)
        if metric_name in val_metrics:
            writer.add_scalar(f"val/{metric_name}", val_metrics[metric_name], epoch)

    writer.add_scalar("train/learning_rate", current_lr, epoch)

    for metric_name in ("loss", "accuracy", "recall", "f1", "f2"):
        writer.add_scalars(
            f"compare/{metric_name}",
            {"train": train_metrics.get(metric_name, 0.0), "val": val_metrics.get(metric_name, 0.0)},
            epoch,
        )
    writer.flush()


# =====================================================================
# 2. LỚP QUẢN LÝ HUẤN LUYỆN LITEFATTRAINER
# =====================================================================
class LiteFatTrainer:
    """
    Lớp Trainer hướng đối tượng quản lý toàn bộ vòng đời huấn luyện mô hình LiteFat:
    1. Khởi tạo TrainConfig và kiểm tra thiết bị.
    2. Xây dựng DataLoaders trực tiếp (không dùng cache).
    3. Khởi tạo mô hình LiteFat, hàm mất mát LiteFatLoss, Optimizer, Scheduler và Scaler.
    4. Điều phối vòng lặp huấn luyện, tính Loss, Accuracy, Recall, F1, F2.
    5. Lưu toàn bộ các checkpoint (epoch_XX.pt, last.pt) và cập nhật best.pt theo điểm F1.
    6. Ghi log Console, train.log và TensorBoard.
    7. Tự động kiểm thử nghiệm thu trên tập Test bằng checkpoint best.pt.
    """
    seed_everything = staticmethod(seed_everything)
    configure_logging = staticmethod(configure_logging)
    atomic_save = staticmethod(atomic_save)
    compute_metrics = staticmethod(compute_metrics)

    def __init__(self, cfg: Optional[TrainConfig] = None):
        self.cfg = cfg if cfg is not None else TrainConfig()
        self.cfg.__post_init__()

        self.save_dir = Path(self.cfg.save_dir)
        self.save_dir.mkdir(parents=True, exist_ok=True)

        self.log_dir = Path(self.cfg.log_dir) if self.cfg.log_dir else (self.save_dir / "tensorboard")
        self.log_dir.mkdir(parents=True, exist_ok=True)

        self.configure_logging(self.save_dir / "train.log")
        self.seed_everything(self.cfg.seed)

        self.device = torch.device(self.cfg.device if torch.cuda.is_available() or self.cfg.device == "cpu" else "cpu")
        self.writer: Optional[SummaryWriter] = None

        self.cfg.save(self.save_dir / "config.json")
        self._setup_components()

    def _setup_components(self) -> None:
        logger.info("=" * 70)
        logger.info("KHỞI TẠO PIPELINE HUẤN LUYỆN LITEFAT (OOP) - DRIVER GUARDIAN")
        logger.info("=" * 70)
        logger.info(f"Thiết bị sử dụng (Device)      : {self.device}")
        logger.info(f"Số lớp phân loại (Classes)     : {self.cfg.num_classes} ({self.cfg.class_names})")
        logger.info(f"Kích thước letterbox           : {self.cfg.letterbox_size}")
        logger.info(f"Kích thước khuôn mặt cắt       : {self.cfg.face_size}")
        logger.info(f"Cửa sổ thời gian (window_sec)  : {self.cfg.window_sec}s | time_interval: {self.cfg.time_interval}s")
        logger.info(f"Chế độ Cache                   : KHÔNG SỬ DỤNG CACHE (Trích xuất On-The-Fly)")
        logger.info(f"Thư mục lưu Checkpoints        : {self.save_dir}")
        logger.info(f"Thư mục log TensorBoard        : {self.log_dir}")

        self.train_loader, self.val_loader, self.test_loader = self._build_dataloaders()
        logger.info(
            f"Số lượng mẫu Dataset           : Train={len(self.train_loader.dataset)} | "
            f"Val={len(self.val_loader.dataset)} | Test={len(self.test_loader.dataset)}"
        )

        self.model = self._build_model().to(self.device)
        total_p = sum(p.numel() for p in self.model.parameters())
        trainable_p = sum(p.numel() for p in self.model.parameters() if p.requires_grad)
        logger.info(f"Tổng số tham số mô hình       : {total_p:,} (Khả học: {trainable_p:,})")

        self.criterion = LiteFatLoss(label_smoothing=self.cfg.label_smoothing)
        self.val_criterion = LiteFatLoss(label_smoothing=0.0)

        self.optimizer = AdamW(
            (p for p in self.model.parameters() if p.requires_grad),
            lr=self.cfg.lr,
            weight_decay=self.cfg.weight_decay,
        )

        self.scheduler = self._build_scheduler()
        self.scaler = torch.amp.GradScaler("cuda", enabled=self.cfg.amp and self.device.type == "cuda")

        self.start_epoch = 1
        self.best_epoch = 0
        self.best_metrics: Optional[Dict[str, float]] = None

        if self.cfg.resume:
            self.load_checkpoint(self.cfg.resume)

    def _build_dataloaders(self):
        return build_litefat_dataloaders(**self.cfg.get_dataloader_kwargs())

    def _build_model(self) -> LiteFat:
        return LiteFat(**self.cfg.get_model_kwargs())

    def _build_scheduler(self) -> LambdaLR:
        return get_cosine_warmup_scheduler(
            self.optimizer,
            total_epochs=self.cfg.epochs,
            warmup_epochs=self.cfg.warmup_epochs,
            min_lr_ratio=self.cfg.min_lr_ratio,
        )

    def load_checkpoint(self, resume_path: str | Path) -> None:
        resume_path = Path(resume_path)
        if not resume_path.is_file():
            logger.warning(f"Không tìm thấy file checkpoint resume tại: {resume_path}. Bỏ qua nạp lại.")
            return

        logger.info(f"Nạp lại checkpoint từ: {resume_path}")
        checkpoint = torch.load(resume_path, map_location=self.device, weights_only=False)
        self.model.load_state_dict(checkpoint["model_state_dict"])
        if "optimizer_state_dict" in checkpoint:
            self.optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
        if "scheduler_state_dict" in checkpoint:
            self.scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
        if "scaler_state_dict" in checkpoint and self.scaler is not None and checkpoint["scaler_state_dict"] is not None:
            self.scaler.load_state_dict(checkpoint["scaler_state_dict"])
        self.start_epoch = checkpoint.get("epoch", 0) + 1
        self.best_epoch = checkpoint.get("best_epoch", 0)
        self.best_metrics = checkpoint.get("best_metrics", None)
        logger.info(f"Đã khôi phục thành công! Tiếp tục từ Epoch {self.start_epoch} (Best Epoch trước đó: {self.best_epoch})")

    def save_checkpoints(
            self,
            epoch: int,
            train_metrics: Dict[str, float],
            val_metrics: Dict[str, float],
            is_best: bool,
    ) -> None:
        """Lưu trữ tất cả các checkpoint: epoch_XX.pt, last.pt và best.pt."""
        checkpoint_state = {
            "format_version": 1,
            "epoch": epoch,
            "model_name": "LiteFat",
            "model_state_dict": self.model.state_dict(),
            "optimizer_state_dict": self.optimizer.state_dict(),
            "scheduler_state_dict": self.scheduler.state_dict(),
            "scaler_state_dict": self.scaler.state_dict() if self.scaler else None,
            "train_metrics": train_metrics,
            "val_metrics": val_metrics,
            "best_epoch": self.best_epoch if not is_best else epoch,
            "best_metrics": self.best_metrics if not is_best else val_metrics,
            "config": self.cfg.to_dict(),
        }

        epoch_ckpt_path = self.save_dir / f"epoch_{epoch:02d}.pt"
        self.atomic_save(checkpoint_state, epoch_ckpt_path)
        self.atomic_save(checkpoint_state, self.save_dir / "last.pt")

        if is_best:
            self.best_epoch = epoch
            self.best_metrics = val_metrics
            best_state = {
                "format_version": 1,
                "epoch": self.best_epoch,
                "model_name": "LiteFat",
                "model_state_dict": {k: v.detach().cpu().clone() for k, v in self.model.state_dict().items()},
                "train_metrics": train_metrics,
                "val_metrics": self.best_metrics,
                "config": self.cfg.to_dict(),
            }
            self.atomic_save(best_state, self.save_dir / "best.pt")
            with open(self.save_dir / "best_metrics.json", "w", encoding="utf-8") as f:
                json.dump(self.best_metrics, f, indent=4, ensure_ascii=False)

    def log_epoch_to_tensorboard(
            self,
            train_metrics: Dict[str, float],
            val_metrics: Dict[str, float],
            current_lr: float,
            epoch: int,
    ) -> None:
        if self.writer is None:
            return
        log_epoch_to_tensorboard(self.writer, train_metrics, val_metrics, current_lr, epoch)

    def run_epoch(
            self,
            loader: DataLoader,
            training: bool = True,
            desc: str = "Train",
    ) -> Dict[str, float]:
        return run_epoch(
            model=self.model,
            loader=loader,
            criterion=self.criterion if training else self.val_criterion,
            device=self.device,
            optimizer=self.optimizer if training else None,
            scaler=self.scaler if training else None,
            amp=self.cfg.amp,
            max_grad_norm=self.cfg.max_grad_norm,
            desc=desc,
            num_classes=self.cfg.num_classes,
        )

    def train_one_epoch(self, epoch: int) -> Dict[str, float]:
        # Tự động mở khóa backbone nếu hết số epoch freeze định trước
        if self.cfg.freeze_backbone_epochs > 0:
            if epoch <= self.cfg.freeze_backbone_epochs and not self.model.is_backbone_frozen:
                logger.info(f"[Backbone] Đóng băng MobileNetV3 backbone tại Epoch {epoch}")
                self.model.freeze_backbone(freeze=True)
            elif epoch > self.cfg.freeze_backbone_epochs and self.model.is_backbone_frozen:
                logger.info(f"[Backbone] Đã hết {self.cfg.freeze_backbone_epochs} epoch warmup -> Mở khóa (Unfreeze) toàn bộ backbone!")
                self.model.unfreeze_backbone()

        return self.run_epoch(
            loader=self.train_loader,
            training=True,
            desc=f"Epoch {epoch:02d}/{self.cfg.epochs:02d} [Train]",
        )

    def validate(self, epoch: int) -> Dict[str, float]:
        return self.run_epoch(
            loader=self.val_loader,
            training=False,
            desc=f"Epoch {epoch:02d}/{self.cfg.epochs:02d} [Val]",
        )

    def evaluate_test(self) -> Optional[Dict[str, float]]:
        best_pt_path = self.save_dir / "best.pt"
        if not best_pt_path.is_file() or len(self.test_loader.dataset) == 0:
            return None

        logger.info("\nĐang tải best.pt để đánh giá nghiệm thu trên tập Test...")
        checkpoint = torch.load(best_pt_path, map_location=self.device, weights_only=False)
        self.model.load_state_dict(checkpoint["model_state_dict"])

        test_metrics = self.run_epoch(
            loader=self.test_loader,
            training=False,
            desc="Final Test",
        )

        if self.writer is not None:
            for m_name, m_val in test_metrics.items():
                self.writer.add_scalar(f"test/{m_name}", m_val, self.best_epoch)
            self.writer.flush()

        with open(self.save_dir / "test_metrics.json", "w", encoding="utf-8") as f:
            json.dump(test_metrics, f, indent=4, ensure_ascii=False)

        logger.info(
            f"Kết quả Test (Best Epoch {self.best_epoch}): "
            f"Loss={test_metrics['loss']:.4f} | Acc={test_metrics['accuracy']:.4f} | "
            f"Recall={test_metrics['recall']:.4f} | F1={test_metrics['f1']:.4f} | F2={test_metrics['f2']:.4f}"
        )
        return test_metrics

    def fit(self) -> nn.Module:
        self.writer = SummaryWriter(str(self.log_dir))
        self.writer.add_text(
            "Config",
            f"```json\n{json.dumps(self.cfg.to_dict(), indent=2, ensure_ascii=False)}\n```",
            0
        )

        logger.info("=" * 70)
        logger.info("BẮT ĐẦU VÒNG LẶP HUẤN LUYỆN (FIT LOOP)")
        logger.info("=" * 70)

        start_train_time = time.time()

        try:
            for epoch in range(self.start_epoch, self.cfg.epochs + 1):
                epoch_start = time.time()
                current_lr = float(self.optimizer.param_groups[0]["lr"])

                train_metrics = self.train_one_epoch(epoch)
                val_metrics = self.validate(epoch)

                self.scheduler.step()
                epoch_duration = time.time() - epoch_start

                self.log_epoch_to_tensorboard(train_metrics, val_metrics, current_lr, epoch)

                # Quyết định checkpoint best dựa trên điểm F1 của tập Val
                is_best = False
                if self.best_metrics is None:
                    is_best = True
                else:
                    val_score = (val_metrics["f1"], val_metrics["f2"], -val_metrics["loss"])
                    best_score = (self.best_metrics["f1"], self.best_metrics["f2"], -self.best_metrics["loss"])
                    if val_score > best_score:
                        is_best = True

                self.save_checkpoints(epoch, train_metrics, val_metrics, is_best)

                best_tag = " >>> [BEST RECORD UPDATED (Val F1)] <<<" if is_best else ""
                logger.info(
                    f"Epoch {epoch:02d}/{self.cfg.epochs:02d} ({epoch_duration:.1f}s) | LR: {current_lr:.6g} [Checkpoint: epoch_{epoch:02d}.pt]\n"
                    f"  - Train: Loss={train_metrics['loss']:.4f} | Acc={train_metrics['accuracy']:.4f} | Recall={train_metrics['recall']:.4f} | F1={train_metrics['f1']:.4f} | F2={train_metrics['f2']:.4f}\n"
                    f"  - Val  : Loss={val_metrics['loss']:.4f} | Acc={val_metrics['accuracy']:.4f} | Recall={val_metrics['recall']:.4f} | F1={val_metrics['f1']:.4f} | F2={val_metrics['f2']:.4f}{best_tag}"
                )

        except KeyboardInterrupt:
            logger.warning("\nQuá trình huấn luyện bị ngắt bởi người dùng. Toàn bộ checkpoint đã được bảo lưu an toàn.")
        finally:
            total_time = time.time() - start_train_time
            logger.info(f"\nHoàn tất huấn luyện sau {total_time / 60:.2f} phút.")
            if self.best_metrics:
                logger.info(f"Checkpoint tốt nhất tại Epoch {self.best_epoch}:")
                logger.info(f"  - Val F1      : {self.best_metrics['f1']:.4f} (Tiêu chí quyết định best)")
                logger.info(f"  - Val F2      : {self.best_metrics['f2']:.4f}")
                logger.info(f"  - Val Recall  : {self.best_metrics['recall']:.4f}")
                logger.info(f"  - Val Accuracy: {self.best_metrics['accuracy']:.4f}")
                logger.info(f"  - Val Loss    : {self.best_metrics['loss']:.4f}")

            self.evaluate_test()

            if self.writer is not None:
                self.writer.close()

        return self.model

In [8]:
# Cell 8: Khởi tạo cấu hình và chạy huấn luyện mô hình LiteFat
# Tự động gán đường dẫn dataset tải từ KaggleHub
dataset_dir_path = str(detected_dataset_dir) if ('detected_dataset_dir' in globals() and detected_dataset_dir is not None) else "./kaggle/data-processed-v3/data_processed"

config = TrainConfig(
)

trainer = LiteFatTrainer(config)
trained_model = trainer.fit()

[2026-11-04 16:40:52][INFO] Epoch 01/20 (5329.8s) | LR: 3.33333e-05 [Checkpoint: epoch_01.pt]
  - Train: Loss=0.4267 | Acc=0.8204 | Recall=0.6980 | F1=0.7882 | F2=0.7315
  - Val  : Loss=0.9965 | Acc=0.5927 | Recall=0.5371 | F1=0.5595 | F2=0.5459 >>> [BEST RECORD UPDATED (Val F1)] <<<


[2026-11-04 18:06:55][INFO] Epoch 02/20 (5162.3s) | LR: 6.66667e-05 [Checkpoint: epoch_02.pt]
  - Train: Loss=0.2081 | Acc=0.9153 | Recall=0.8909 | F1=0.9097 | F2=0.8983
  - Val  : Loss=1.4129 | Acc=0.5138 | Recall=0.5276 | F1=0.5111 | F2=0.5209


Epoch 03/20 [Train]:  95%|█████████▌| 310/326 [1:08:11<03:27, 12.98s/batch, loss=0.1846]

: 

In [ ]:
# Cell 9: Khởi chạy TensorBoard trực tiếp trên Colab & Nén kết quả checkpoints để tải về
%load_ext tensorboard
%tensorboard --logdir LiteFat/checkpoints/tensorboard

# Nén toàn bộ checkpoints, log và file cấu hình thành litefat_results.zip
!zip -r litefat_results.zip LiteFat/checkpoints
print("Đã nén toàn bộ kết quả thành file litefat_results.zip. Bạn có thể tải file này về máy!")